# 🏡 Chatbot Bumi Nirwana Residence
**Stack:** Streamlit · Anthropic Claude · PyMuPDF · Ngrok

### Alur kerja:
```
Upload PDF  →  Ekstrak teks  →  Jadikan system prompt  →  Streamlit UI  →  Ngrok (URL publik)
```

### Langkah:
1. Install library
2. Set API Key
3. Upload & ekstrak PDF
4. Tulis `app.py`
5. Jalankan Streamlit + Ngrok

## 📦 Langkah 1 — Install Library

In [2]:
!pip install streamlit anthropic pymupdf pyngrok --quiet
print('✅ Semua library berhasil diinstall!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 81.8 MB/s eta 0:00:00
✅ Semua library berhasil diinstall!


## 🔑 Langkah 2 — Masukkan API Key
- Anthropic API Key → https://console.anthropic.com
- Ngrok Auth Token  → https://dashboard.ngrok.com

In [3]:
import os
from getpass import getpass

ANTHROPIC_API_KEY = getpass('🔑 Anthropic API Key: ')
NGROK_AUTH_TOKEN  = getpass('🔑 Ngrok Auth Token : ')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
print('✅ API Key tersimpan!')

🔑 Anthropic API Key: ··········
🔑 Ngrok Auth Token : ··········
✅ API Key tersimpan!


## 📄 Langkah 3 — Upload & Ekstrak PDF

Jalankan sel ini, lalu klik tombol **Choose Files** yang muncul dan pilih file PDF dataset Bumi Nirwana Residence kamu.

In [5]:
from google.colab import files
import fitz  # PyMuPDF
import os

print('📂 Silakan upload file PDF dataset kamu...')
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f'\n✅ File diterima: {pdf_filename}')

# Ekstrak seluruh teks dari PDF
doc = fitz.open(pdf_filename)
extracted_text = ''
for page_num, page in enumerate(doc, start=1):
    extracted_text += f'\n--- Halaman {page_num} ---\n'
    extracted_text += page.get_text()

print(f'✅ Berhasil mengekstrak {len(extracted_text):,} karakter dari {len(doc)} halaman')
print('\n📋 Preview 500 karakter pertama:')
print('-' * 50)
print(extracted_text[:500])
print('-' * 50)

doc.close()

# Simpan ke file teks agar bisa dibaca app.py
with open('dataset_bnr.txt', 'w', encoding='utf-8') as f:
    f.write(extracted_text)

print('\n✅ Dataset tersimpan ke dataset_bnr.txt')

📂 Silakan upload file PDF dataset kamu...


Saving dataset.pdf to dataset (1).pdf

✅ File diterima: dataset (1).pdf
✅ Berhasil mengekstrak 9,421 karakter dari 4 halaman

📋 Preview 500 karakter pertama:
--------------------------------------------------

--- Halaman 1 ---
 1 
Pertanyaan: 
1) Perumahan Bumi Nirwana Residence berada di mana? 
2) Dimana alamat perumahan Bumi Nirwana Residence? 
3) Dimana lokasi Perumahan Bumi Nirwana Residence? 
4) Bagaimana rute menuju lokasi perumahan dari pusat kota? 
 
Jawaban: 
1) Perumahan Bumi Nirwana Residence berlokasi di wilayah Lumajang. Untuk detail alamat 
dan titik lokasi, silakan lihat menu Lokasi atau hubungi marketing. 
2) Perumahan Bumi Nirwana Residence (Sumberejo) berlokasi di Sumberejo, Kec. 

--------------------------------------------------

✅ Dataset tersimpan ke dataset_bnr.txt


## 🖥️ Langkah 4 — Tulis File app.py (Aplikasi Chatbot)

In [15]:
kode_app = r'''
import streamlit as st
import anthropic
import os

st.set_page_config(
    page_title="Bumi Nirwana Residence — Asisten Properti",
    page_icon="🏡",
    layout="centered"
)

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600&family=Inter:wght@400;500&display=swap');
    html, body, .stApp { background-color: #F8F6F1 !important; font-family: 'Inter', sans-serif; }
    .bnr-header { background: linear-gradient(135deg, #1B4332 0%, #2D6A4F 100%); color: white; padding: 24px 28px 20px; border-radius: 16px; margin-bottom: 20px; }
    .bnr-header h1 { font-family: 'Playfair Display', serif; font-size: 22px; margin: 0 0 4px 0; color: white !important; }
    .bnr-header p { font-size: 13px; margin: 0; color: #A8D5BA; }
    .bnr-badge { display: inline-block; background: #D4AF37; color: #1B2505; font-size: 11px; font-weight: 600; padding: 3px 10px; border-radius: 999px; margin-bottom: 10px; }
    .bubble-user { background: #2D6A4F; color: white; padding: 11px 16px; border-radius: 16px 16px 4px 16px; margin: 6px 0; max-width: 78%; margin-left: auto; font-size: 14px; line-height: 1.5; word-wrap: break-word; }
    .bubble-bot { background: white; border: 1px solid #E2DDD4; color: #1a1a1a; padding: 11px 16px; border-radius: 16px 16px 16px 4px; margin: 6px 0; max-width: 78%; font-size: 14px; line-height: 1.5; word-wrap: break-word; }
    .label-user { text-align:right; color:#2D6A4F; font-size:11px; margin-bottom:2px; font-weight:500; }
    .label-bot { color:#888; font-size:11px; margin-bottom:2px; }
    .stTextInput > div > div > input { background: white !important; border: 1px solid #C8C3BB !important; border-radius: 10px !important; font-size: 14px !important; color: #1a1a1a !important; }
    .stButton > button { background: #2D6A4F !important; color: white !important; border: none !important; border-radius: 10px !important; font-weight: 500 !important; width: 100% !important; padding: 10px !important; }
    .stButton > button:hover { background: #1B4332 !important; }
    section[data-testid="stSidebar"] { background: #F0EDE6 !important; }
</style>
""", unsafe_allow_html=True)


# ── Muat dataset ─────────────────────────────────────────────────────────────
@st.cache_resource
def muat_dataset():
    if os.path.exists('dataset_bnr.txt'):
        with open('dataset_bnr.txt', 'r', encoding='utf-8') as f:
            return f.read()
    return None

dataset_text = muat_dataset()

SYSTEM_PROMPT = """Kamu adalah asisten virtual resmi Bumi Nirwana Residence.
Jawab pertanyaan calon pembeli dengan sopan, ramah, dan profesional dalam Bahasa Indonesia.
Gunakan HANYA informasi dari dataset berikut. Jika tidak ada, katakan jujur dan sarankan hubungi tim marketing.
Jangan mengarang informasi.

--- DATASET ---
{dataset}
--- AKHIR DATASET ---
""".format(dataset=dataset_text or "Dataset tidak tersedia.")


# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 🏡 Bumi Nirwana Residence")
    st.markdown("*Asisten Properti Digital*")
    st.divider()

    model_pilihan = st.selectbox(
        "Model AI",
        ["claude-haiku-4-5-20251001", "claude-sonnet-4-5"],
        index=0
    )
    max_token = st.slider("Panjang jawaban", 256, 2048, 1024, step=256)

    st.divider()
    if dataset_text:
        st.success(f"✅ Dataset aktif ({len(dataset_text):,} karakter)")
    else:
        st.error("❌ Dataset tidak ditemukan!")

    st.divider()
    st.markdown("**Pertanyaan cepat:**")
    pertanyaan_list = [
        "Harga unit tersedia?",
        "Cara KPR di sini?",
        "Fasilitas apa saja?",
        "Lokasi & akses jalan?",
        "Jadwal kunjungan?",
    ]
    for pq in pertanyaan_list:
        if st.button(pq, key=f"q_{pq}"):
            st.session_state['kirim_pesan'] = pq

    st.divider()
    if st.button("🗑️ Hapus riwayat"):
        st.session_state.messages = []
        st.rerun()

    st.caption("© Bumi Nirwana Residence · Claude AI")


# ── Header ────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="bnr-header">
    <div class="bnr-badge">✦ PROPERTI PREMIUM</div>
    <h1>🏡 Bumi Nirwana Residence</h1>
    <p>Asisten virtual kami siap membantu Anda menemukan hunian impian.</p>
</div>
""", unsafe_allow_html=True)


# ── Inisialisasi session ──────────────────────────────────────────────────────
if 'messages' not in st.session_state:
    st.session_state.messages = []
if 'kirim_pesan' not in st.session_state:
    st.session_state.kirim_pesan = ''


# ── Tampilkan riwayat ─────────────────────────────────────────────────────────
if not st.session_state.messages:
    st.markdown("""
    <div style="text-align:center; padding:30px 0; color:#888;">
        <div style="font-size:40px; margin-bottom:10px;">🏘️</div>
        <p style="font-size:15px; color:#555;">Selamat datang di Bumi Nirwana Residence!</p>
        <p style="font-size:13px;">Tanyakan apa saja tentang perumahan kami.</p>
    </div>
    """, unsafe_allow_html=True)
else:
    for msg in st.session_state.messages:
        if msg['role'] == 'user':
            st.markdown('<div class="label-user">Anda</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="bubble-user">{msg["content"]}</div>', unsafe_allow_html=True)
        else:
            st.markdown('<div class="label-bot">🏡 Asisten BNR</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="bubble-bot">{msg["content"]}</div>', unsafe_allow_html=True)


# ── Input form — pakai st.form agar Enter & tombol Kirim sama-sama berfungsi ──
with st.form(key='form_chat', clear_on_submit=True):
    col1, col2 = st.columns([5, 1])
    with col1:
        user_input = st.text_input(
            label='pesan',
            value=st.session_state.get('kirim_pesan', ''),
            placeholder='Tanyakan harga, fasilitas, lokasi, KPR...',
            label_visibility='collapsed'
        )
    with col2:
        kirim = st.form_submit_button('Kirim ➤')


# ── Proses jawaban ────────────────────────────────────────────────────────────
# Reset quick input
if st.session_state.kirim_pesan:
    st.session_state.kirim_pesan = ''

if kirim and user_input and user_input.strip():
    teks = user_input.strip()

    # Simpan pesan user
    st.session_state.messages.append({'role': 'user', 'content': teks})

    # Panggil Claude
    try:
        api_key = os.environ.get('ANTHROPIC_API_KEY', '')
        if not api_key:
            st.error('❌ ANTHROPIC_API_KEY tidak ditemukan. Jalankan ulang Langkah 2.')
            st.stop()

        client = anthropic.Anthropic(api_key=api_key)

        with st.spinner('🏡 Asisten sedang mencari informasi...'):
            response = client.messages.create(
                model=model_pilihan,
                max_tokens=max_token,
                system=SYSTEM_PROMPT,
                messages=[
                    {'role': m['role'], 'content': m['content']}
                    for m in st.session_state.messages
                ]
            )

        balasan = response.content[0].text
        st.session_state.messages.append({'role': 'assistant', 'content': balasan})

    except anthropic.AuthenticationError:
        st.error('❌ API Key tidak valid. Cek kembali di console.anthropic.com')
    except anthropic.RateLimitError:
        st.error('⚠️ Batas penggunaan API tercapai. Tunggu beberapa saat.')
    except Exception as e:
        st.error(f'❌ Error: {str(e)}')

    st.rerun()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(kode_app)

print('✅ app.py berhasil diperbarui!')
print('👉 Sekarang jalankan ulang Langkah 5 untuk restart Streamlit.')

✅ app.py berhasil diperbarui!
👉 Sekarang jalankan ulang Langkah 5 untuk restart Streamlit.


## 🚀 Langkah 5 — Fungsi Helper & Jalankan Chatbot

In [13]:
import subprocess
import time
import os
from pyngrok import ngrok, conf
from getpass import getpass

# ── Set token langsung di sini (lebih aman) ──────────────────────
NGROK_TOKEN = getpass('🔑 Ngrok Auth Token: ')

def jalankan_chatbot_bnr(nama_file='app.py', port=8501):

    # Validasi file
    for file in [nama_file, 'dataset_bnr.txt']:
        if not os.path.exists(file):
            print(f'❌ File {file} tidak ditemukan!')
            return None
    print('✅ Semua file ditemukan')

    # Set Ngrok token SEBELUM connect (ini yang penting!)
    conf.get_default().auth_token = NGROK_TOKEN
    print('✅ Ngrok token berhasil diset')

    # Tutup tunnel lama jika ada
    try:
        ngrok.kill()
        time.sleep(2)
        print('✅ Tunnel lama ditutup')
    except Exception:
        pass

    # Jalankan Streamlit
    print(f'🚀 Menjalankan Streamlit di port {port}...')
    subprocess.Popen(
        [
            'streamlit', 'run', nama_file,
            '--server.port', str(port),
            '--server.headless', 'true',
            '--server.enableCORS', 'false',
            '--server.enableXsrfProtection', 'false',
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(5)
    print(f'✅ Streamlit berjalan di localhost:{port}')

    # Buka tunnel Ngrok
    try:
        tunnel     = ngrok.connect(port)
        url_publik = tunnel.public_url
        print()
        print('=' * 60)
        print('🏡  CHATBOT BUMI NIRWANA RESIDENCE SIAP!')
        print(f'🔗  URL Publik : {url_publik}')
        print('=' * 60)
        return url_publik

    except Exception as e:
        print(f'❌ Gagal membuka tunnel Ngrok: {e}')
        print('💡 Cek token di dashboard.ngrok.com → Your Authtoken')
        return None

# Jalankan
url = jalankan_chatbot_bnr()

🔑 Ngrok Auth Token: ··········
✅ Semua file ditemukan
✅ Ngrok token berhasil diset
✅ Tunnel lama ditutup
🚀 Menjalankan Streamlit di port 8501...
✅ Streamlit berjalan di localhost:8501

🏡  CHATBOT BUMI NIRWANA RESIDENCE SIAP!
🔗  URL Publik : https://stunning-family-armadillo.ngrok-free.dev


## 💡 Tips & Troubleshooting

| Masalah | Solusi |
|---|---|
| `AuthenticationError` | Cek Anthropic API Key, pastikan tidak expired |
| URL Ngrok tidak bisa dibuka | Cek Ngrok Auth Token, atau restart sel Langkah 5 |
| Dataset tidak terbaca | Pastikan Langkah 3 sudah dijalankan dan sukses |
| Bot menjawab tidak relevan | Cek isi PDF — pastikan teks ter-ekstrak dengan benar |
| Port sudah dipakai | Ganti `port=8501` menjadi `8502` atau `8503` |
| Chatbot berhenti sendiri | Colab auto-disconnect — aktifkan GPU atau gunakan Colab Pro |

### Cara lihat isi dataset yang ter-ekstrak:
```python
with open('dataset_bnr.txt', 'r') as f:
    print(f.read())
```

### Cara stop chatbot:
```python
from pyngrok import ngrok
ngrok.kill()
print('Chatbot dihentikan.')
```

In [16]:
import os, anthropic, subprocess, requests, time

print('=' * 50)
print('DIAGNOSIS CHATBOT BNR')
print('=' * 50)

# 1. Cek API Key
api_key = os.environ.get('ANTHROPIC_API_KEY', '')
print(f'\n[1] API Key : {"✅ Ada" if api_key else "❌ TIDAK ADA"}')

# 2. Cek Dataset
ada = os.path.exists('dataset_bnr.txt')
print(f'[2] Dataset : {"✅ Ada" if ada else "❌ TIDAK ADA"}')
if ada:
    with open('dataset_bnr.txt') as f:
        isi = f.read()
    print(f'    Ukuran  : {len(isi):,} karakter')

# 3. Cek Streamlit
hasil = subprocess.run(['pgrep','-f','streamlit'], capture_output=True, text=True)
print(f'[3] Streamlit: {"✅ Berjalan" if hasil.stdout.strip() else "❌ MATI"}')

# 4. Test Claude API langsung
print('\n[4] Test Claude API...')
try:
    client = anthropic.Anthropic(api_key=api_key)
    r = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=100,
        messages=[{'role':'user','content':'Halo, jawab: satu tambah satu berapa?'}]
    )
    print(f'    ✅ Claude menjawab: {r.content[0].text}')
except Exception as e:
    print(f'    ❌ Error: {e}')

# 5. Cek isi app.py — apakah pakai st.form?
print('\n[5] Cek app.py...')
if os.path.exists('app.py'):
    with open('app.py') as f:
        kode = f.read()
    print(f'    st.form     : {"✅ Ada" if "st.form" in kode else "❌ TIDAK ADA"}')
    print(f'    ANTHROPIC   : {"✅ Ada" if "ANTHROPIC_API_KEY" in kode else "❌ TIDAK ADA"}')
    print(f'    rerun       : {"✅ Ada" if "st.rerun" in kode else "❌ TIDAK ADA"}')
else:
    print('    ❌ app.py tidak ditemukan!')

print('\n' + '=' * 50)
print('Paste hasil di atas ke chat!')
print('=' * 50)

DIAGNOSIS CHATBOT BNR

[1] API Key : ✅ Ada
[2] Dataset : ✅ Ada
    Ukuran  : 9,421 karakter
[3] Streamlit: ✅ Berjalan

[4] Test Claude API...
    ❌ Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaxFK9qBB8gK9fHxFCXkB'}

[5] Cek app.py...
    st.form     : ✅ Ada
    ANTHROPIC   : ✅ Ada
    rerun       : ✅ Ada

Paste hasil di atas ke chat!
